```yaml
title: Replacing your USASpending integration with Tango
description: Same questions you ask USASpending today — answered with one SDK call instead of a JSON-walk over a paginated POST.
tags: [migration, contracts, joining, business-development]
endpoints: [list_contracts]
```

# Replacing your USASpending integration with Tango

If you're building on federal procurement data today, odds are high you've already written a USASpending integration: a `POST` to `/api/v2/search/spending_by_award/`, a filter envelope full of nested arrays, a pagination loop, and a JSON-walk to pull out the fields you actually wanted. It works, until you need to *join* anything — vehicles to their pool benches, IDVs to their task orders, UEIs across endpoints. Then, you're maintaining your own normalization layer forever.

Tango has the federated, normalized version of the same data. This notebook takes two queries every USASpending integration ends up writing and shows the Tango equivalent. If you recognize your own code in the "before" snippets, the "after" is a few SDK calls away.

## Query 1 — "recent IT-services awards"

The job: pull the latest contract awards in NAICS 541512 (Computer Systems Design Services) since March, sorted newest first. Standard BD / capture sweep.

### The USASpending way

```python
import requests

url = "https://api.usaspending.gov/api/v2/search/spending_by_award/"
payload = {
    "filters": {
        "award_type_codes": ["A", "B", "C", "D"],
        "naics_codes": ["541512"],
        "time_period": [{"start_date": "2026-03-01", "end_date": "2026-12-31"}],
    },
    "fields": [
        "Award ID", "Recipient Name", "recipient_id",
        "Award Amount", "Start Date",
        "Awarding Agency", "Awarding Sub Agency",
    ],
    "sort": "Start Date",
    "order": "desc",
    "limit": 5,
    "page": 1,
}

rows = requests.post(url, json=payload).json()["results"]
for r in rows:
    print(r["Start Date"], r["Award Amount"],
          r["Recipient Name"], r["Awarding Agency"])
```

Three things you end up needing to know: which `award_type_codes` mean "contracts" (vs. grants, vs. loans), where the `time_period` envelope lives, and which of the 80+ field names returns the thing you actually wanted. Pagination is via `page`/`limit`. Joins on the recipient require their separate `recipient/{id}/` endpoint. Set-aside, PSC, vehicle relationships, and protests live in different endpoints with different shapes.

In [1]:
import requests

url = "https://api.usaspending.gov/api/v2/search/spending_by_award/"
payload = {
    "filters": {
        "award_type_codes": ["A", "B", "C", "D"],
        "naics_codes": ["541512"],
        "time_period": [{"start_date": "2026-03-01", "end_date": "2026-12-31"}],
    },
    "fields": [
        "Award ID", "Recipient Name", "recipient_id",
        "Award Amount", "Base Obligation Date",
        "Awarding Agency", "Awarding Sub Agency",
    ],
    "sort": "Base Obligation Date",
    "order": "desc",
    "limit": 5,
    "page": 1,
}

rows = requests.post(url, json=payload).json()["results"]
for r in rows:
    print(r["Base Obligation Date"], r["Award Amount"],
          r["Recipient Name"], r["Awarding Agency"])

2026-06-05 198915.52 CDW GOVERNMENT LLC Department of Transportation
2026-06-05 1659603.66 KONIAG TECHNOLOGY SOLUTIONS INC Department of Agriculture
2026-06-04 3080692.59 ATTAINX INC. Department of Homeland Security
2026-06-04 50414.4 SHARP SOLUTIONS INC Nuclear Regulatory Commission
2026-06-04 17790.26 BCT PARTNERS LLC Department of Justice


### The Tango way

Same question, one call:

In [2]:
import os
from tango import TangoClient

tango = TangoClient(api_key=os.environ["TANGO_API_KEY"])

# `shape=` is response shaping — ask for what you want, get a trimmed row.
# Use `field(*)` to pull all sub-fields of a nested object.
SHAPE = (
    "key,piid,award_date,obligated,"
    "recipient(display_name,uei),"
    "awarding_office(*)"
)

contracts = tango.list_contracts(
    naics_code="541512",
    award_date_gte="2026-03-01",
    ordering="-award_date",
    shape=SHAPE,
    limit=5,
)

for c in contracts.results:
    obligated = float(c["obligated"])
    print(f"{c['award_date']}  ${obligated:>14,.2f}  "
          f"{c['recipient']['display_name']:<32}  "
          f"{c['awarding_office']['agency_name']}")

2026-06-05  $  1,659,603.66  KONIAG TECHNOLOGY SOLUTIONS INC   ANIMAL AND PLANT HEALTH INSPECTION SERVICE
2026-06-05  $    198,915.52  CDW GOVERNMENT LLC                NATIONAL HIGHWAY TRAFFIC SAFETY ADMINISTRATION
2026-06-04  $     17,790.26  BCT PARTNERS LLC                  FEDERAL PRISON SYSTEM / BUREAU OF PRISONS
2026-06-04  $  3,080,692.59  ATTAINX INC.                      OFFICE OF PROCUREMENT OPERATIONS
2026-06-04  $    223,563.64  ELECTROSOFT SERVICES, LLC         OFFICE OF THE SECRETARY


Same answer, less envelope. A few things changed under the surface:

- **No award-type code lookup.** `list_contracts` only returns contracts. Grants, IDVs, OTAs, and subawards are their own endpoints. The thing you mean is the endpoint you call.
- **Filters are kwargs, not a nested object.** `naics_code="541512"` instead of `"filters": {"naics_codes": ["541512"]}`. Pagination is `cursor=`, not page-counting.
- **The recipient is a structured object on the row.** `c['recipient']['uei']` and `c['recipient']['display_name']` are already there — no second call to a recipient endpoint, no UEI-to-name reconciliation on your side.
- **`shape=` controls what comes back.** Ask for what you need; skip what you don't. The minimal default is even smaller than what we asked for here.

## Query 2 — "what's riding on OASIS+, by pool?"

The natural BD follow-up that's awkward against USASpending: vehicles aren't first-class in FPDS. "OASIS+" is six parent contracts wearing one name; the task orders riding underneath are transaction rows pointing at one of those parents.

### The USASpending way

To answer "give me OASIS+ broken out by pool, with bench size and TO count," you'd start by finding the parent IDV awards. `spending_by_award` returns them — but flat, with no pool grouping:

In [3]:
url = "https://api.usaspending.gov/api/v2/search/spending_by_award/"
payload = {
    "filters": {
        "award_type_codes": ["IDV_B", "IDV_B_A", "IDV_B_B", "IDV_B_C"],
        "agencies": [{"type": "awarding", "tier": "subtier",
                       "name": "Federal Acquisition Service"}],
        "psc_codes": ["R499"],   # Other Professional Services — OASIS+'s PSC
        "time_period": [{"start_date": "2024-01-01", "end_date": "2026-12-31",
                          "date_type": "new_awards_only"}],
    },
    "fields": ["Award ID", "Recipient Name", "Recipient UEI", "Award Amount"],
    "sort": "Award Amount", "order": "desc", "limit": 5, "page": 1,
}
resp = requests.post(url, json=payload).json()

for row in resp["results"]:
    print(f"{row['Award ID']:<18}  ${float(row['Award Amount']):>10,.0f}  "
          f"{row['Recipient UEI']:<14}  {row['Recipient Name']}")
print()
print(f"... (hasNext={resp['page_metadata']['hasNext']}; "
      f"there are hundreds more awardee IDVs in this filter alone)")

47QRCA26DV019       $     2,500  SDAPDLJ5EZ33    GREEN POWERED TECHNOLOGY LLC
47QRCA26DW013       $     2,500  HN8PMK8CE559    PRAESCIENT ANALYTICS LLC
47QRCA26DV014       $     2,500  D1VPKJAJZW36    GOLDSTONE PARTNERS LLC
47QRCA26DSG60       $     2,500  E71KZAMT5MD8    HELIOS DEFENSE SOLUTIONS LLC
47QRCA26DSG59       $     2,500  MNPNZE1FEW54    CANVAS II, LLC

... (hasNext=True; there are hundreds more awardee IDVs in this filter alone)


Every row is one awardee's IDV under OASIS+. Which pool (SBA / 8(a) / SDVOSB / WOSB / HUBZone / Unrestricted) does each belong to? USASpending doesn't tell you — the pool is encoded in the parent solicitation, which isn't on the award row. To roll up to pool-level metrics you'd:

1. Look up the six OASIS+ parent solicitations (out-of-band).
2. Six more `spending_by_award` searches with `parent_award_piid` set to each, count distinct UEIs for bench, sum obligations.
3. Six more searches for delivery orders against each parent.

You'll get there. Twelve POSTs, hand-curated pool reconciliation, no awardee-bench endpoint (USAS only shows awardees who've *won* a TO; the silent bench is invisible), and you're rebuilding the same script every time a new vehicle launches.

### The Tango way

Vehicles are first-class. One row per pool, bench size and TO count already on it:

In [4]:
pools = tango.list_vehicles(
    search="OASIS+",
    award_date_after="2024-01-01",
    shape=(
        "uuid,solicitation_identifier,set_aside,"
        "awardee_count,order_count,total_obligated,vehicle_contracts_value"
    ),
    ordering="-order_count",
    limit=10,
)

for v in sorted(pools.results, key=lambda r: r["solicitation_identifier"]):
    obligated = float(v["total_obligated"]) / 1e6
    ceiling   = float(v["vehicle_contracts_value"]) / 1e6
    print(f"{v['solicitation_identifier']}  {(v['set_aside'] or '—'):<8}  "
          f"awardees={v['awardee_count']:>4}  TOs={v['order_count']:>4}  "
          f"${obligated:>7.1f}M obligated  ${ceiling:>7.1f}M ceiling")

47QRCA23R0001  SBA       awardees= 102  TOs= 142  $  356.3M obligated  $ 2313.2M ceiling
47QRCA23R0002  8A        awardees=  35  TOs=  44  $  101.7M obligated  $  632.6M ceiling
47QRCA23R0003  HZC       awardees=   5  TOs=   5  $    5.4M obligated  $   28.0M ceiling
47QRCA23R0004  SDVOSBC   awardees=  27  TOs=  37  $   95.9M obligated  $  418.9M ceiling
47QRCA23R0005  WOSB      awardees=  10  TOs=  13  $   11.3M obligated  $   59.9M ceiling
47QRCA23R0006  —         awardees=  48  TOs=  72  $  253.1M obligated  $ 2646.1M ceiling


Drill into any one of those pools for recent task orders:

In [5]:
sb = next(v for v in pools.results if v["set_aside"] == "SBA")

orders = tango.list_vehicle_orders(
    uuid=sb["uuid"],
    shape="key,award_date,obligated,awarding_office(*),description",
    limit=20,
)

# Server-side ordering on this sub-resource isn't honored yet; sort client-side.
for o in sorted(orders.results, key=lambda r: r["award_date"], reverse=True)[:5]:
    desc = (o["description"] or "").replace("\n", " ")[:46]
    agency = o["awarding_office"]["agency_name"][:30]
    print(f"{o['award_date']}  ${float(o['obligated'])/1e6:>6.2f}M  "
          f"{agency:<32}  {desc}")

2026-06-03  $  9.17M  OFFICE OF THE SECRETARY           ANTIDUMPING (AD) AND COUNTERVAILING DUTY (CVD)
2026-05-12  $  0.15M  ENVIRONMENTAL PROTECTION AGENC    R7 SSP: HASTINGS COLORADO AVENUE OU01 GROUNDWA
2026-05-12  $  0.10M  DEPARTMENTAL OFFICES              CARPENTRY SUPPORT SERVICES
2026-05-08  $  0.27M  OFFICE OF PROCUREMENT OPERATIO    THIS REQUIREMENT SUPPORTS THE AGENCIES ORGANIZ
2026-04-30  $  0.65M  FEDERAL EMERGENCY MANAGEMENT A    NDEMI INDEPENDENT STUDY PROGRAM SUPPORT SERVIC


A few structural things changed:

- **Vehicle is a first-class object.** `list_vehicles(...)` returns one row per pool — solicitation ID, set-aside, bench size, TO count, obligated dollars, and ceiling — already joined. In USASpending these live across IDV awards, transactions, and a base-award-document blob you parse yourself.
- **Pool bench is on the vehicle.** `awardee_count` is on the row; `list_vehicle_awardees(uuid=...)` gives the full bench, including awardees who haven't won a TO yet. USASpending only shows you the ones who *have* won — the silent bench is invisible.
- **Task orders are the vehicle's children.** `list_vehicle_orders(uuid=...)` filters by parent automatically. In USASpending that's a `parent_award_piid` filter against `spending_by_award` — fine if you already know the PIID; the prior six POSTs are what got you the PIID.

Same join semantics for OTA vehicles via `list_otidvs` / `get_otidv` — the SDK keeps it consistent across vehicle types.

## What you'd build with this

- **A capture-side recompete watcher.** Combine `list_contracts(naics_code=..., expiring_gte=..., expiring_lte=...)` with the [`saved-search-watcher`](../examples/saved-search-watcher/) pattern to alert on contracts about to expire in your NAICS — without writing the date arithmetic across USASpending's `period_of_performance` envelope.
- **A vendor-mix dashboard for any agency.** `list_contracts(awarding_agency=..., award_date_gte=...)` returns rows you can group by `recipient.uei` in a single pass. No `recipient/{id}/` follow-ups, no name-collision dedup.
- **Drop your USASpending MCP server.** If you've built an MCP around USASpending, the equivalent in Tango is one tool per endpoint, vehicles as first-class objects with their pool bench and task orders already joined, and the same UEI across every dataset. The [`tango-mcp`](https://github.com/makegov/tango-mcp) server ships it all.
- **Skip the next 200 lines of pagination glue.** `cursor`-based pagination, configurable timeouts, and shape-controlled responses are the SDK defaults.

The shortest version of the pitch: **anything you can ask USASpending, you can ask Tango — plus the joins you used to maintain yourself**. The [free tier](https://tango.makegov.com) is enough to A/B your existing integration against one Tango call and see what changes.